# 02 — Limpeza

**Objetivo:** Recarregar o raw, padronizar nomes de colunas (snake_case PT-BR via `src/data.py`), mover `id_cliente` para índice e validar integridade dos dados. Salva `dados_limpos.parquet`.

**Inputs:** `data/raw/dataset.csv`

**Outputs:** `data/processed/dados_limpos.parquet`

---

**Roteiro:**

1. Setup, tema e configurações locais e globais
2. Carregamendo dos dados
3. Persistir dados limpos

### Etapa 1 — Setup, tema e configurações locais e globais

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import CONFIG, CAMINHOS
from src.data import padronizar_colunas

DADOS_BRUTOS = CAMINHOS.dados_raw / CAMINHOS.dados_brutos
DADOS_PRO = CAMINHOS.dados_processed
DADOS_PRO.mkdir(parents=True, exist_ok=True)

print("✅ Setup OK\n")
print(f"DADOS_BRUTOS: {DADOS_BRUTOS}")
print(f"DADOS_PRO   : {DADOS_PRO}")

✅ Setup OK

DADOS_BRUTOS: C:\Users\jas_t\Repo\Portfolios\customer_segmentation_mall\data\raw\Mall_Customers.csv
DADOS_PRO   : C:\Users\jas_t\Repo\Portfolios\customer_segmentation_mall\data\processed


### Etapa 2 - Carregamendo dos dados

In [2]:
df = pd.read_csv(DADOS_BRUTOS)
df = padronizar_colunas(df)
df = df.set_index("id_cliente")

print(f"   Shape: {df.shape[0]} linhas × {df.shape[1]} colunas\n")
df.head()

   Shape: 200 linhas × 4 colunas



,genero,idade,renda_anual,score_gasto
id_cliente,,,,
1,Male,19,15,39
2,Male,21,15,81
3,Female,20,16,6
4,Female,23,16,77
5,Female,31,17,40


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 1 to 200
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   genero       200 non-null    str  
 1   idade        200 non-null    int64
 2   renda_anual  200 non-null    int64
 3   score_gasto  200 non-null    int64
dtypes: int64(3), str(1)
memory usage: 7.4 KB


In [4]:
df.describe().round(2)

,idade,renda_anual,score_gasto
count,200.00,200.00,200.00
mean,38.85,60.56,50.20
std,13.97,26.26,25.82
min,18.00,15.00,1.00
25%,28.75,41.50,34.75
50%,36.00,61.50,50.00
75%,49.00,78.00,73.00
max,70.00,137.00,99.00


In [5]:
df.describe(exclude="number")

,genero
count,200
unique,2
top,Female
freq,112


### Célula 3 — Persistir dados limpos

In [6]:
SAIDA = DADOS_PRO / "dados_limpos.parquet"

df.to_parquet(SAIDA, index=True)

print(f"   Salvo: {SAIDA.relative_to(CAMINHOS.dados_processed.parent.parent)}")
print(f"   Shape: {df.shape[0]} linhas × {df.shape[1]} colunas")

   Salvo: data\processed\dados_limpos.parquet
   Shape: 200 linhas × 4 colunas


In [7]:
df_check = pd.read_parquet(SAIDA)
idx_ok   = df_check.index.name == "id_cliente"
shape_ok = df_check.shape == df.shape
cols_ok  = list(df_check.columns) == list(df.columns)

print("\n   Verificação round-trip:")
print(f"   {'✅' if idx_ok else '❌'} índice id_cliente preservado")
print(f"   {'✅' if shape_ok else '❌'} shape idêntico após releitura")
print(f"   {'✅' if cols_ok else '❌'} colunas e ordem preservadas")


   Verificação round-trip:
   ✅ índice id_cliente preservado
   ✅ shape idêntico após releitura
   ✅ colunas e ordem preservadas
